In [37]:
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [38]:
# collect missing data and remove data with complete rate < 0.8 (threshold set to 10781)
missing_stats = pd.read_csv('data/missing_value_data.csv')
cols_to_remove = missing_stats[missing_stats['Missing Count'] > 10781]['Variable Name'].tolist()

In [39]:
# read data
yougov = pd.read_csv("raw_data/australia.csv",na_values=[" ", "__NA__"], keep_default_na=True)

In [40]:
# drop high missing data
yougov = yougov.drop(columns=[col for col in cols_to_remove if col in yougov.columns], errors='ignore')

In [41]:
# convert datetime
yougov['endtime'] = pd.to_datetime(yougov['endtime'], dayfirst=True).dt.floor('D')

# dates for consent for medical questions was not given and set time mask
start_date, end_date = "2021-02-10", "2021-10-18"
time_mask = yougov['endtime'].between(start_date, end_date)

# define columns that require special filling
target_cols = [f"PHQ4_{i}" for i in range(1, 5)] + \
              [f"d1_health_{i}" for i in list(range(1, 14)) + [98, 99]]

# fill in the missing values within a specific time period
for col in target_cols:
    if col in yougov.columns:
        yougov.loc[time_mask, col] = yougov.loc[time_mask, col].fillna("N/A")

In [42]:
# initial clean
yougov.dropna(inplace=True)

# remapping r1 to value
r1_mapping = {"1 – Disagree": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7 - Agree": 7}
yougov[['r1_1', 'r1_2']] = yougov[['r1_1', 'r1_2']].replace(r1_mapping)

# mapping i12 text to value
freq_mapping = {"Not at all": 1, "Rarely": 2, "Sometimes": 3, "Frequently": 4, "Always": 5}
i12_cols = [c for c in yougov.columns if c.startswith("i12_health_")]
yougov[i12_cols] = yougov[i12_cols].replace(freq_mapping)

In [43]:
# mask-wearing binary classification
mask_cols = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
yougov["face_mask_behaviour_scale"] = yougov[mask_cols].median(axis=1)
yougov["face_mask_behaviour_binary"] = np.where(yougov["face_mask_behaviour_scale"] >= 4, "Yes", "No")

# protective behaviour
all_protect_cols = yougov.filter(like="i12_").columns
yougov["protective_behaviour_scale"] = yougov[all_protect_cols].median(axis=1)
yougov["protective_behaviour_binary"] = np.where(yougov["protective_behaviour_scale"] >= 4, "Yes", "No")

# protective behaviour except mask
nomask_cols = list(set(all_protect_cols) - set(mask_cols))
yougov["protective_behaviour_nomask_scale"] = yougov[nomask_cols].median(axis=1)

# comorbidities
yougov["d1_comorbidities"] = "Yes"
yougov.loc[yougov["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"
yougov.loc[yougov["d1_health_99"] == "N/A", "d1_comorbidities"] = "NA"
yougov.loc[yougov["d1_health_98"] == "Yes", "d1_comorbidities"] = "Prefer_not_to_say"

# drop the original d1 column
yougov = yougov.drop(columns=[c for c in yougov.columns if c.startswith("d1_health")], axis=1)

In [44]:
# recalculate the 14-day cycle
min_date = yougov['endtime'].min()
yougov['week_number'] = ((yougov['endtime'] - min_date).dt.days // 14) + 1

# convert household
household_map = {str(i): i for i in range(1, 8)}
household_map.update({"8 or more": 8, "Prefer not to say": np.nan, "Don't know": np.nan})
yougov["household_size"] = yougov["household_size"].map(household_map)

# last clean
yougov.dropna(inplace=True)

In [45]:
# define the columns to be dropped
cols_to_drop = ["qweek", "weight"] + i12_cols
yougov.drop(columns=cols_to_drop, inplace=True)

# save cleaned data
yougov.to_csv("data/cleaned_data.csv", index=False)